# Quickstart: deep hedging vs delta on real Hong Kong stocks

This notebook reads the published per-window results (no training needed, ~10 s) and reproduces the headline comparison. To retrain everything, run `scripts/run_all.sh` (~3 h on 2 CPUs).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apiwittri2007/DeepHedging_Project/blob/main/notebooks/quickstart.ipynb)

In [ ]:
import os, sys
if 'google.colab' in sys.modules and not os.path.exists('DeepHedging_Project'):
    !git clone -q https://github.com/apiwittri2007/DeepHedging_Project
    %cd DeepHedging_Project
import pandas as pd, numpy as np
pnl = pd.read_csv('results/training_data/pnl_by_window.csv.gz')
pnl.head()

## CVaR₉₅ by arm and transaction cost
CVaR₉₅ = average loss in the worst 5% of the 2,650 test windows (HK$ per HK$100 of stock). Deep-hedger arms average over seeds.

In [ ]:
seed_cols = [c for c in pnl.columns if c.startswith('pnl_seed')]
def cvar(x, a=0.95):
    x = np.sort(np.asarray(x)); k = max(1, int(np.ceil((1-a)*len(x)))); return -x[:k].mean()
rows = []
for (arm, cost), g in pnl.groupby(['arm', 'cost']):
    vals = [cvar(g[c].dropna()) for c in seed_cols if g[c].notna().any()]
    rows.append({'arm': arm, 'cost_bps': round(cost*1e4), 'cvar': np.mean(vals)})
table = pd.DataFrame(rows).pivot(index='arm', columns='cost_bps', values='cvar').round(3)
table.sort_values(15)

## Gap to BS delta as costs rise
Negative = less tail risk than delta. Significance tests (month-clustered bootstrap, Holm and Romano–Wolf corrections) are in `results/training_data/multiple_testing.csv`.

In [ ]:
import matplotlib.pyplot as plt
gap = table.sub(table.loc['delta'], axis=1).drop(['delta', 'no_hedge'], errors='ignore')
ax = gap.T.plot(marker='o', figsize=(8, 4.5))
ax.axhline(0, color='k', lw=1); ax.set_xlabel('cost per side (bps)'); ax.set_ylabel('CVaR − CVaR(delta)')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8); plt.tight_layout()

In [ ]:
pd.read_csv('results/training_data/multiple_testing.csv').query('cost == 0.0015').round({'diff': 3, 'ci_low': 3, 'ci_high': 3, 'p_raw': 3, 'p_holm': 3, 'p_romano_wolf': 3})